In [ ]:
import copy
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import rl_mind.envs
from dataclasses import dataclass, replace
from scipy import stats
from torch import Tensor
import gymnasium as gym
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from rl_mind.core import Action, Actor
from rl_mind.nn import build_mlp, soft_update
from rl_mind.env import VecEnv
from rl_mind.data import ReplayBuffer, Transitions
from rl_mind.collectors import TransitionCollector
from rl_mind.evaluation import Evaluator
from rl_mind.notebook import run_directory, setup_tensorboard, silence_known_warnings


In [ ]:
silence_known_warnings()

In [ ]:
env = gym.make("LunarLander-v3", continuous=True)

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

obs, info = env.reset(seed=0)
print("Example observation:", obs)
print("Example random action:", env.action_space.sample())

In [ ]:
def add_layernorm(model: nn.Module):
    layers = list(model.children())
    linear_indices = [i for i, layer in enumerate(layers)if isinstance(layer, nn.Linear)]
    last_linear_index = linear_indices[-1]
    new_layers = []
    for i, layer in enumerate(layers):
        new_layers.append(layer)
        if isinstance(layer, nn.Linear) and i != last_linear_index:
            new_layers.append(nn.LayerNorm(layer.out_features))
    return nn.Sequential(*new_layers)

In [ ]:
class ContinuousQNetwork(nn.Module):
    def __init__(self, obs_dim: int, hidden: tuple[int, ...], action_dim: int, use_layernorm: bool = False):
        super().__init__()
        self.model = build_mlp([obs_dim + action_dim, *hidden, 1])
        if use_layernorm:
            self.model = add_layernorm(self.model)

    def forward(self, obs: Tensor, action: Tensor) -> Tensor:
        return self.model(torch.cat([obs, action], dim=1)).squeeze(-1)

In [ ]:
class ContinuousDeterministicActor(Actor[Action]):
    def __init__(self, obs_dim: int, hidden: tuple[int, ...], action_dim: int, use_layernorm: bool = False):
        super().__init__()
        self.model = build_mlp(
            [obs_dim, *hidden, action_dim], output_activation=nn.Tanh()
        )
        if use_layernorm:
            self.model = add_layernorm(self.model)

    def forward(self, obs: Tensor) -> Action:
        return Action(value=self.model(obs))

In [ ]:
class GaussianNoise(Actor[Action]):
    def __init__(self, actor: Actor[Action], sigma: float):
        super().__init__()
        self.actor = actor
        self.sigma = sigma

    def forward(self, obs: Tensor) -> Action:
      action = self.actor(obs).value
      noisy_action = action + self.sigma * torch.randn_like(action)
      return Action(value=noisy_action.clamp(-1.0, 1.0))

    def act(self, obs: Tensor) -> Tensor:
        return self.actor.act(obs)

In [ ]:
def compute_critic_loss(
    gamma: float, batch: Transitions[Action], q_values: Tensor, next_q_values: Tensor
) -> Tensor:
    target = (batch.reward+ gamma * (~batch.terminated) * next_q_values)
    return F.mse_loss(q_values, target)

In [ ]:
def compute_actor_loss(q_values: Tensor) -> Tensor:
    target = -q_values.mean()
    return target

In [ ]:
@dataclass(frozen=True)
class DDPGConfig:
    env_name: str = "LunarLander-v3"
    seed: int = 1

    use_layernorm: bool = False

    max_steps: int = 30_000
    n_envs: int = 1
    steps_per_update: int = 1
    learning_starts: int = 1_000

    buffer_size: int = 200_000
    batch_size: int = 64

    gamma: float = 0.98
    tau: float = 0.05
    action_noise: float = 0.1

    actor_hidden: tuple[int, ...] = (64, 64)
    critic_hidden: tuple[int, ...] = (64, 64)
    lr_actor: float = 1e-3
    lr_critic: float = 1e-3

    eval_interval: int = 2_000
    n_eval_envs: int = 10

In [ ]:
def run_ddpg(cfg: DDPGConfig) -> Evaluator:
    torch.manual_seed(cfg.seed)
    env = VecEnv(cfg.env_name,cfg.n_envs,seed=cfg.seed,continuous=True)

    actor = ContinuousDeterministicActor(
        env.observation_dim, cfg.actor_hidden, env.action_dim, use_layernorm=cfg.use_layernorm
    )
    critic = ContinuousQNetwork(
        env.observation_dim, cfg.critic_hidden, env.action_dim, use_layernorm=cfg.use_layernorm
    )
    target_critic = copy.deepcopy(critic)
    target_actor = copy.deepcopy(actor)

    actor_optimizer = torch.optim.Adam(actor.parameters(), lr=cfg.lr_actor)
    critic_optimizer = torch.optim.Adam(critic.parameters(), lr=cfg.lr_critic)

    collector = TransitionCollector(env, GaussianNoise(actor, cfg.action_noise))
    buffer = ReplayBuffer(cfg.buffer_size)
    ln_name = "LN" if cfg.use_layernorm else "noLN"
    run_dir = run_directory(f"ddpg-{cfg.env_name}-{ln_name}-S{cfg.seed}")
    evaluator = Evaluator(
        VecEnv(cfg.env_name,cfg.n_eval_envs,seed=cfg.seed + 100,continuous=True),
        every=cfg.eval_interval,
        run_dir=run_dir,
        writer=SummaryWriter(run_dir),
    )

    pbar = tqdm(total=cfg.max_steps)
    while collector.steps < cfg.max_steps:
        buffer.add(collector.collect(cfg.steps_per_update))
        pbar.update(collector.steps - pbar.n)
        if len(buffer) < cfg.learning_starts:
            continue

        batch = buffer.sample(cfg.batch_size)

        q_values = critic(batch.obs, batch.action.value)
        with torch.no_grad():
            next_actions = target_actor(batch.next_obs).value
            next_q_values = target_critic(batch.next_obs, next_actions)

        critic_loss = compute_critic_loss(cfg.gamma, batch, q_values, next_q_values)
        critic_optimizer.zero_grad()
        critic_loss.backward()
        critic_optimizer.step()

        actions = actor(batch.obs).value
        q_values = critic(batch.obs, actions)
        actor_loss = compute_actor_loss(q_values)

        actor_optimizer.zero_grad()
        actor_loss.backward()
        actor_optimizer.step()

        soft_update(critic, target_critic, cfg.tau)
        soft_update(actor, target_actor, cfg.tau)

        evaluator.writer.add_scalar("loss/critic", critic_loss.item(), collector.steps)
        evaluator.writer.add_scalar("loss/actor", actor_loss.item(), collector.steps)
        if result := evaluator.run_if_needed(collector.steps, actor):
            pbar.set_description(
                f"eval={result.mean:7.1f} best={evaluator.best_reward:7.1f}"
            )
            mc_env = gym.make(cfg.env_name, continuous=True)
            obs, _ = mc_env.reset(seed=cfg.seed + 1000)

            mc_return = 0.0
            discount = 1.0
            q_pred = None
            done = False

            while not done:
                state = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)

                with torch.no_grad():
                    action = actor(state).value
                    if q_pred is None:
                        q_pred = critic(state, action).item()

                obs, reward, terminated, truncated, _ = mc_env.step(
                    action.squeeze(0).numpy()
                )

                mc_return += discount * reward
                discount *= cfg.gamma
                done = terminated or truncated

            mc_env.close()

            evaluator.writer.add_scalar(
                "Q_vs_MC/Q_value",
                q_pred,
                collector.steps,
            )
            evaluator.writer.add_scalar(
                "Q_vs_MC/Monte_Carlo",
                mc_return,
                collector.steps,
            )
            evaluator.writer.add_scalar(
                "Q_vs_MC/Bias",
                q_pred - mc_return,
                collector.steps,
            )

            evaluator.writer.flush()

            print(
                f"Step {collector.steps}: "
                f"Q={q_pred:.2f}, "
                f"MC={mc_return:.2f}, "
                f"Bias={q_pred - mc_return:.2f}"
            )
    pbar.close()
    return evaluator

In [ ]:
setup_tensorboard()

In [ ]:
# Experiment 1
for i in range(5):
  ddpg_config = DDPGConfig(
      seed=i,
      use_layernorm=False,
  )
  ddpg_no_ln = run_ddpg(ddpg_config)

In [ ]:
# Experiment 2
for i in range(5):
  ddpg_config = DDPGConfig(
      seed=i,
      use_layernorm=True,
  )
  ddpg_ln = run_ddpg(ddpg_config)